In [6]:
pip install cartopy netCDF4 rasterio opencv-python satpy

  Using cached donfig-0.8.1.post1-py3-none-any.whl.metadata (5.0 kB)
  Using cached configobj-5.0.9-py2.py3-none-any.whl.metadata (3.2 kB)
   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ---------------------------------------- 1.9/1.9 MB 20.5 MB/s  0:00:00
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ------------------------------------ --- 2.1/2.3 MB 9.9 MB/s eta 0:00:01
   ---------------------------------------- 2.3/2.3 MB 6.9 MB/s  0:00:00
Using cached configobj-5.0.9-py2.py3-none-any.whl (35 kB)
Using cached donfig-0.8.1.post1-py3-none-any.whl (21 kB)
   ---------------------------------------- 0.0/799.3 kB ? eta -:--:--
   ---------------------------------------- 799.3/799.3 kB 40.0 MB/s  0:00:00

   -------- -------------------------------  3/14 [pathlib-abc]
   ----------- ----------------------------  4/14 [numcodecs]
   ---------------------- -----------------  8/14 [zarr]
   ---------------------- -----------------  8/14 [z

In [4]:
"""
get_mask.py
===========
Módulo de reclassificação de máscaras de nuvem a partir de arquivos NetCDF4
do produto ABI-L2-ACMF (Clear Sky Mask) do satélite GOES-16.

Fluxo principal:
    1. `NetCDFBatchProcessor`  – leitura e inspeção de arquivos NetCDF4.
    2. `ImageBatchProcessor`   – reclassificação e redimensionamento de arrays.
    3. `GeoTIFFBatchExporter`  – exportação georreferenciada para GeoTIFF.
    4. `batch_process_files`   – orquestra o pipeline completo com suporte a
                                  processamento paralelo.

Reclassificação padrão (produto ACMF):
    0 → nuvem presente  (original: céu claro)
    1 → céu claro       (original: nuvem)
    A inversão é intencional: valor 1 marca pixels válidos (sem nuvem).

Dependências:
    affine, numpy, xarray, rasterio, opencv-python

Uso típico:
    batch_process_files(
        product='ABI-L2-ACMF',
        output_dir='Arquivos/MASCARA_RECLASSIFICADA',
        base_path='Dados',
    )
"""

import os
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Dict, List, Optional, Tuple

import numpy as np
import xarray as xr
import rasterio
from rasterio.crs import CRS
import cv2
from affine import Affine

# ---------------------------------------------------------------------------
# Constantes do módulo
# ---------------------------------------------------------------------------

# Mapeamento de reclassificação padrão para o produto ACMF:
# 0 (nuvem) → mantém 0 | 1 (céu claro) → mantém 1 (invertido para máscara)
DEFAULT_RECLASS_MAP: Dict[int, int] = {1: 0, 0: 1}

# Resolução padrão Full Disk do ABI (pixels)
DEFAULT_TARGET_SHAPE: Tuple[int, int] = (5424, 5424)

# Variável alvo para leitura via satpy no produto ACMF
ACMF_VARIABLE = "BCM"

# Valor sentinela para pixels sem dado no GeoTIFF de saída
NODATA_VALUE: float = np.nan


# ---------------------------------------------------------------------------
# Classes de processamento
# ---------------------------------------------------------------------------

class NetCDFBatchProcessor:
    """
    Leitura e inspeção de arquivos NetCDF4 de produtos GOES-16.

    Parâmetros:
        base_path (str): Diretório raiz onde os produtos estão armazenados.
                         Esperado: <base_path>/<product>/netCDF/*.nc
    """

    def __init__(self, base_path: str):
        if not os.path.isdir(base_path):
            raise FileNotFoundError(f"Diretório base não encontrado: '{base_path}'")
        self.base_path = base_path
        print(f"Base path configurado: {self.base_path}")

    def _product_dir(self, product: str) -> str:
        """Retorna o caminho completo do diretório netCDF de um produto."""
        return os.path.join(self.base_path, product, "netCDF")

    def list_product_files(self, product: str) -> List[str]:
        """
        Lista todos os arquivos NetCDF4 disponíveis para um produto.

        Parâmetros:
            product (str): Código do produto (ex.: 'ABI-L2-ACMF').

        Retorna:
            list[str]: Nomes dos arquivos `.nc` ordenados alfabeticamente.

        Levanta:
            FileNotFoundError: Se o diretório do produto não existir.
        """
        product_dir = self._product_dir(product)
        if not os.path.isdir(product_dir):
            raise FileNotFoundError(
                f"Diretório do produto não encontrado: '{product_dir}'"
            )
        return sorted(f for f in os.listdir(product_dir) if f.endswith(".nc"))

    def get_variables(self, product: str, filename: str) -> Dict[int, str]:
        """
        Retorna as variáveis disponíveis em um arquivo NetCDF4.

        Parâmetros:
            product  (str): Código do produto.
            filename (str): Nome do arquivo `.nc`.

        Retorna:
            dict[int, str]: Mapeamento {índice: nome_da_variável}.
        """
        file_path = os.path.join(self._product_dir(product), filename)
        # xarray é usado de forma consistente em todo o módulo (sem netCDF4 diretamente)
        with xr.open_dataset(file_path) as ds:
            return {i: var for i, var in enumerate(ds.variables.keys())}

    def load_dataset(self, product: str, filename: str) -> xr.Dataset:
        """
        Carrega um arquivo NetCDF4 como xarray Dataset.

        O dataset é retornado aberto; o chamador é responsável por fechá-lo
        (preferencialmente com bloco `with`).

        Parâmetros:
            product  (str): Código do produto.
            filename (str): Nome do arquivo `.nc`.

        Retorna:
            xr.Dataset: Dataset carregado.
        """
        file_path = os.path.join(self._product_dir(product), filename)
        return xr.open_dataset(file_path)


class ImageBatchProcessor:
    """Operações vetorizadas de reclassificação e redimensionamento de arrays 2D."""

    @staticmethod
    def reclassify_array(
        data: np.ndarray,
        mapping: Dict[int, int],
    ) -> np.ndarray:
        """
        Reclassifica um array numpy de acordo com um dicionário de mapeamento.

        Pixels cujo valor não esteja no mapeamento recebem 0.

        Parâmetros:
            data    (np.ndarray): Array de entrada (inteiros).
            mapping (dict):       Dicionário {valor_original: novo_valor}.

        Retorna:
            np.ndarray: Array reclassificado, dtype float32.

        Exemplos:
            >>> reclassify_array(arr, {0: 1, 1: 0})  # inverte máscara binária
        """
        result = np.zeros(data.shape, dtype=np.float32)
        for original, new in mapping.items():
            result[data == original] = new
        return result

    @staticmethod
    def to_float32(data: np.ndarray) -> np.ndarray:
        """
        Converte um array para float32 de forma segura e vetorizada.

        Valores que não podem ser convertidos (ex.: strings) tornam-se NaN.

        Parâmetros:
            data (np.ndarray): Array de entrada.

        Retorna:
            np.ndarray: Array convertido para float32.
        """
        try:
            return data.astype(np.float32)
        except (ValueError, TypeError):
            # Fallback para arrays com tipos mistos ou objetos não numéricos
            return np.where(
                np.vectorize(lambda x: isinstance(x, (int, float)))(data),
                data.astype(np.float32),
                np.nan,
            ).astype(np.float32)

    @staticmethod
    def resize_image(
        image: np.ndarray,
        target_shape: Tuple[int, int] = DEFAULT_TARGET_SHAPE,
    ) -> np.ndarray:
        """
        Redimensiona um array 2D para o shape alvo usando interpolação nearest-neighbor.

        Interpolação nearest-neighbor preserva os valores discretos da máscara
        (evita a criação de valores intermediários inválidos).

        Parâmetros:
            image        (np.ndarray): Array 2D de entrada.
            target_shape (tuple):      Dimensões alvo (largura, altura) em pixels.

        Retorna:
            np.ndarray: Array redimensionado.
        """
        return cv2.resize(image, target_shape, interpolation=cv2.INTER_NEAREST)


class GeoTIFFBatchExporter:
    """Exportação georreferenciada de arrays numpy para o formato GeoTIFF."""

    @staticmethod
    def create_georef(file_path: str):
        """
        Extrai transformação afim e CRS diretamente dos atributos de projeção
        geoestacionária do NetCDF (sem dependência do satpy).

        O GOES-16 armazena os parâmetros de projeção na variável
        'goes_imager_projection' seguindo a convenção CF. O CRS é construído
        manualmente como projeção Geostationary com os parâmetros do satélite.

        Parâmetros:
            file_path (str): Caminho completo do arquivo NetCDF4.

        Retorna:
            tuple: (Affine, CRS) — transformação afim e sistema de referência.
        """
        with xr.open_dataset(file_path) as ds:
            proj_var = ds["goes_imager_projection"]

            # Parâmetros orbitais e de projeção do GOES-16
            lon_origin    = float(proj_var.attrs["longitude_of_projection_origin"])
            persp_height  = float(proj_var.attrs["perspective_point_height"])
            semi_major    = float(proj_var.attrs["semi_major_axis"])
            semi_minor    = float(proj_var.attrs["semi_minor_axis"])
            sweep         = str(proj_var.attrs.get("sweep_angle_axis", "x"))

            # Coordenadas angulares em radianos → metros na projeção
            x_rad = ds["x"].values * persp_height   # radianos × altura → metros
            y_rad = ds["y"].values * persp_height

            res_x =  float(x_rad[1] - x_rad[0])
            res_y =  float(y_rad[1] - y_rad[0])   # negativo: y decresce para baixo
            x_min =  float(x_rad[0])
            y_max =  float(y_rad[0])

            transform = Affine.translation(x_min, y_max) * Affine.scale(res_x, res_y)

            # CRS como projeção Geostationary (PROJ string)
            crs = CRS.from_proj4(
                f"+proj=geos +lon_0={lon_origin} +h={persp_height} "
                f"+a={semi_major} +b={semi_minor} +sweep={sweep} +units=m"
            )

        return transform, crs

    @staticmethod
    def export_to_geotiff(
        data: np.ndarray,
        crs,
        transform: Affine,
        output_path: str,
        nodata: float = NODATA_VALUE,
    ) -> None:
        """
        Exporta um array numpy para um arquivo GeoTIFF georreferenciado.

        Parâmetros:
            data        (np.ndarray): Array 2D a exportar (float32).
            crs:                      Sistema de referência de coordenadas (cartopy CRS).
            transform   (Affine):     Transformação afim do raster.
            output_path (str):        Caminho completo do arquivo de saída.
            nodata      (float):      Valor sentinela para pixels sem dado.
        """
        profile = {
            "driver":    "GTiff",
            "height":    data.shape[0],
            "width":     data.shape[1],
            "count":     1,
            "dtype":     "float32",
            "crs":       crs,
            "transform": transform,
            "nodata":    nodata,
            "compress":  "lzw",    # compressão sem perda para reduzir tamanho em disco
            "tiled":     True,     # acesso eficiente a subconjuntos do raster
        }

        # Garante que o diretório de saída existe antes de tentar escrever
        output_dir = os.path.dirname(output_path)
        if output_dir:
            os.makedirs(output_dir, exist_ok=True)

        with rasterio.open(output_path, "w", **profile) as dst:
            dst.write(data, 1)


# ---------------------------------------------------------------------------
# Funções de orquestração
# ---------------------------------------------------------------------------

def _get_georef_from_file(file_path: str):
    """
    Extrai transformação afim e CRS do primeiro arquivo NetCDF do produto.

    Wrapper de conveniência para `GeoTIFFBatchExporter.create_georef`,
    usado para obter o georreferenciamento uma única vez antes do loop.

    Parâmetros:
        file_path (str): Caminho completo do arquivo NetCDF4.

    Retorna:
        tuple: (Affine, CRS) — transformação afim e sistema de referência.
    """
    return GeoTIFFBatchExporter.create_georef(file_path)


def process_single_file(
    ncdf_processor: NetCDFBatchProcessor,
    img_processor: ImageBatchProcessor,
    exporter: GeoTIFFBatchExporter,
    product: str,
    filename: str,
    reclass_map: Dict[int, int],
    target_shape: Tuple[int, int],
    output_dir: str,
    area,                            # área compartilhada entre arquivos do mesmo produto
) -> Tuple[str, bool, object]:
    """
    Processa um único arquivo NetCDF4 e exporta a máscara reclassificada como GeoTIFF.

    Parâmetros:
        ncdf_processor (NetCDFBatchProcessor): Processador de NetCDF.
        img_processor  (ImageBatchProcessor):  Processador de imagem.
        exporter       (GeoTIFFBatchExporter): Exportador GeoTIFF.
        product        (str): Código do produto.
        filename       (str): Nome do arquivo `.nc`.
        reclass_map    (dict): Mapeamento de reclassificação.
        target_shape   (tuple): Dimensões alvo do raster de saída.
        output_dir     (str): Diretório de saída dos GeoTIFFs.
        area:          Tupla (Affine, CRS) compartilhada entre arquivos do produto.

    Retorna:
        tuple: (filename, sucesso: bool, tempo_ou_erro)
    """
    try:
        start_time = time.time()

        # 1. Leitura dos dados brutos
        with ncdf_processor.load_dataset(product, filename) as ds:
            first_var = next(iter(ds.variables))
            raw_data = ds[first_var].values  # carrega para memória e fecha o dataset

        # 2. Reclassificação e conversão
        reclassified = img_processor.reclassify_array(raw_data, reclass_map)
        float_array  = img_processor.to_float32(reclassified)
        resized      = img_processor.resize_image(float_array, target_shape)

        # 3. Georreferenciamento — transform e crs já pré-calculados
        transform, crs = area

        # 4. Exportação
        output_filename = f"{os.path.splitext(filename)[0]}_RECLASSIFIED.tif"
        output_path     = os.path.join(output_dir, output_filename)
        exporter.export_to_geotiff(resized, crs, transform, output_path)

        elapsed = time.time() - start_time
        return (filename, True, elapsed)

    except Exception as e:
        return (filename, False, str(e))


def _chunked(lst: List, size: int) -> List[List]:
    """
    Divide uma lista em sublistas de tamanho máximo `size`.

    Parâmetros:
        lst  (list): Lista a dividir.
        size (int):  Tamanho máximo de cada chunk.

    Retorna:
        list[list]: Lista de sublistas.
    """
    return [lst[i : i + size] for i in range(0, len(lst), size)]


def batch_process_files(
    product: str,
    output_dir: str,
    base_path: str,
    reclass_map: Dict[int, int] = DEFAULT_RECLASS_MAP,
    target_shape: Tuple[int, int] = DEFAULT_TARGET_SHAPE,
    max_workers: int = 4,
    chunk_size: int = 16,
) -> None:
    """
    Orquestra o processamento em lote de todos os arquivos de um produto.

    Os arquivos são processados em chunks para controlar o consumo de memória:
    em vez de submeter todos os arquivos de uma vez ao executor (o que pré-aloca
    futures para todos simultaneamente), o processamento ocorre em grupos de
    `chunk_size` arquivos. Isso garante que no máximo `chunk_size` arrays
    5424×5424 float32 (~112 MB cada) coexistam na memória.

    A área geográfica é extraída uma única vez do primeiro arquivo e
    reutilizada para todos os demais, evitando chamadas repetidas ao satpy.

    Parâmetros:
        product      (str):  Código do produto (ex.: 'ABI-L2-ACMF').
        output_dir   (str):  Diretório de saída dos GeoTIFFs reclassificados.
        base_path    (str):  Diretório raiz dos dados NetCDF.
        reclass_map  (dict): Mapeamento de reclassificação (padrão: {1:0, 0:1}).
        target_shape (tuple): Dimensões alvo do raster (padrão: 5424×5424).
        max_workers  (int):  Número de threads paralelas por chunk (padrão: 4).
        chunk_size   (int):  Número de arquivos processados por rodada (padrão: 16).
                             Recomendado: chunk_size >= max_workers para manter
                             todas as threads ocupadas dentro do chunk.

    Levanta:
        FileNotFoundError: Se nenhum arquivo for encontrado para o produto.
    """
    # Inicializa os processadores
    ncdf_processor = NetCDFBatchProcessor(base_path)
    img_processor  = ImageBatchProcessor()
    exporter       = GeoTIFFBatchExporter()

    # Lista arquivos disponíveis
    files = ncdf_processor.list_product_files(product)
    if not files:
        raise FileNotFoundError(
            f"Nenhum arquivo NetCDF encontrado para o produto '{product}' em '{base_path}'"
        )

    chunks = _chunked(files, chunk_size)

    print(f"Produto       : {product}")
    print(f"Arquivos      : {len(files)}")
    print(f"Chunks        : {len(chunks)} × até {chunk_size} arquivos")
    print(f"Workers       : {max_workers}")
    print(f"Saída         : {output_dir}\n")

    # Extrai transformação afim e CRS uma única vez a partir do primeiro arquivo
    # Usa atributos de projeção geoestacionária do NetCDF (sem satpy)
    first_file_path = os.path.join(base_path, product, "netCDF", files[0])
    print(f"Extraindo georreferenciamento de: {files[0]}\n")
    area = _get_georef_from_file(first_file_path)

    os.makedirs(output_dir, exist_ok=True)

    success_count = 0

    for chunk_idx, chunk in enumerate(chunks, start=1):
        print(f"── Chunk {chunk_idx}/{len(chunks)} ({len(chunk)} arquivos) ──")

        # Submete apenas os arquivos do chunk atual; libera memória ao final
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {
                executor.submit(
                    process_single_file,
                    ncdf_processor,
                    img_processor,
                    exporter,
                    product,
                    filename,
                    reclass_map,
                    target_shape,
                    output_dir,
                    area,
                ): filename
                for filename in chunk
            }

            for future in as_completed(futures):
                filename, status, result = future.result()
                if status:
                    print(f"  ✔ {filename} — {result:.2f}s")
                    success_count += 1
                else:
                    print(f"  ✘ {filename} — Erro: {result}")

    print(f"\nConcluído: {success_count}/{len(files)} arquivos processados com sucesso.")


# ---------------------------------------------------------------------------
# Exemplo de uso
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    PRODUCT    = "ABI-L2-ACMF"
    BASE_PATH  = "Dados"
    OUTPUT_DIR = os.path.join("MASCARA_RECLASSIFICADA")

    start_total = time.time()

    batch_process_files(
        product=PRODUCT,
        output_dir=OUTPUT_DIR,
        base_path=BASE_PATH,
        max_workers=4,    # aumente conforme os recursos disponíveis
        chunk_size=16,    # arquivos por rodada; chunk_size >= max_workers
    )

    print(f"\nTempo total: {time.time() - start_total:.2f}s")

Base path configurado: Dados
Produto       : ABI-L2-ACMF
Arquivos      : 12
Chunks        : 1 × até 16 arquivos
Workers       : 4
Saída         : MASCARA_RECLASSIFICADA

Extraindo georreferenciamento de: OR_ABI-L2-ACMF-M6_G16_s20201501300166_e20201501309474_c20201501310195.nc

── Chunk 1/1 (12 arquivos) ──
  ✔ OR_ABI-L2-ACMF-M6_G16_s20201501330166_e20201501339474_c20201501340237.nc — 2.07s
  ✔ OR_ABI-L2-ACMF-M6_G16_s20201501310166_e20201501319474_c20201501320211.nc — 2.54s
  ✔ OR_ABI-L2-ACMF-M6_G16_s20201501320166_e20201501329474_c20201501330152.nc — 2.67s
  ✔ OR_ABI-L2-ACMF-M6_G16_s20201501300166_e20201501309474_c20201501310195.nc — 2.66s
  ✔ OR_ABI-L2-ACMF-M6_G16_s20201501340166_e20201501349474_c20201501350251.nc — 1.34s
  ✔ OR_ABI-L2-ACMF-M6_G16_s20201501350166_e20201501359474_c20201501400201.nc — 1.33s
  ✔ OR_ABI-L2-ACMF-M6_G16_s20201511310165_e20201511319473_c20201511320224.nc — 1.27s
  ✔ OR_ABI-L2-ACMF-M6_G16_s20201511300165_e20201511309473_c20201511310160.nc — 1.45s
  ✔ OR_ABI-L